# Rasterization Diagnostics

Validate the rasterization pipeline for CNN input. Key checks:
1. Are branches continuously connected to the confluence under 8-connectivity?
2. What causes gaps — rotation, rounding, or resize?
3. How prevalent is the problem across the dataset?

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pathlib import Path
from scipy import ndimage

from channel_heads.rasterizer import (
    BACKGROUND, BRANCH_A, BRANCH_B, OTHER_STREAMS, CONFLUENCE_MARKER,
    rasterize_outlet_pair, _compute_rotation_angle, _rotate_coordinates, _get_rc,
)
from channel_heads.first_meet_pairs_for_outlet import (
    _build_parents_from_stream,
    _collect_basin_nodes_from_outlet,
    _build_children_from_parents,
)
from channel_heads.geometric_analysis import _trace_full_path
from channel_heads.config import RESULTS_DIR

## 1. Connectivity check utilities

In [ ]:
# 8-connectivity structuring element
STRUCT_8 = np.ones((3, 3), dtype=int)


def check_branch_connectivity(raster, branch_val, conf_val=CONFLUENCE_MARKER):
    """Check if all pixels of a branch are 8-connected to the confluence.
    
    Returns
    -------
    dict with keys:
        connected: bool - is the branch fully connected to confluence?
        n_components: int - number of connected components in (branch | confluence)
        n_branch_pixels: int - total branch pixels
        n_connected_to_conf: int - branch pixels in the same component as confluence
        n_disconnected: int - branch pixels NOT connected to confluence
    """
    # Mask: branch OR confluence pixels
    mask = (raster == branch_val) | (raster == conf_val)
    if not mask.any():
        return {"connected": False, "n_components": 0, "n_branch_pixels": 0,
                "n_connected_to_conf": 0, "n_disconnected": 0}
    
    labeled, n_components = ndimage.label(mask, structure=STRUCT_8)
    
    # Find which component contains the confluence
    conf_mask = raster == conf_val
    if not conf_mask.any():
        return {"connected": False, "n_components": n_components, 
                "n_branch_pixels": int((raster == branch_val).sum()),
                "n_connected_to_conf": 0, 
                "n_disconnected": int((raster == branch_val).sum())}
    
    conf_label = labeled[conf_mask][0]
    
    branch_mask = raster == branch_val
    n_branch = int(branch_mask.sum())
    n_connected = int(((labeled == conf_label) & branch_mask).sum())
    
    return {
        "connected": n_connected == n_branch,
        "n_components": n_components,
        "n_branch_pixels": n_branch,
        "n_connected_to_conf": n_connected,
        "n_disconnected": n_branch - n_connected,
    }


def diagnose_raster(raster):
    """Run full connectivity diagnostics on a raster."""
    diag_a = check_branch_connectivity(raster, BRANCH_A)
    diag_b = check_branch_connectivity(raster, BRANCH_B)
    has_conf = (raster == CONFLUENCE_MARKER).any()
    return {"branch_a": diag_a, "branch_b": diag_b, "has_confluence": has_conf}


print("Diagnostic utilities loaded.")

## 2. Visualization utilities

In [ ]:
CLASS_CMAP = mcolors.ListedColormap(
    ["white", "#1f77b4", "#d62728", "#aaaaaa", "#e8c520"]
)
CLASS_NAMES = ["Background", "Branch A", "Branch B", "Other streams", "Confluence"]
LEGEND_PATCHES = [Patch(color=CLASS_CMAP(i), label=CLASS_NAMES[i]) for i in range(5)]


def plot_raster_with_diagnostics(raster, title="", ax=None):
    """Plot raster with connectivity annotations."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    ax.imshow(raster, cmap=CLASS_CMAP, vmin=0, vmax=4, interpolation="nearest")
    
    diag = diagnose_raster(raster)
    a_ok = diag["branch_a"]["connected"]
    b_ok = diag["branch_b"]["connected"]
    
    status = []
    if a_ok:
        status.append("A: connected")
    else:
        n_disc = diag['branch_a']['n_disconnected']
        n_comp = diag['branch_a']['n_components']
        status.append(f"A: BROKEN ({n_disc} disconnected px, {n_comp} components)")
    
    if b_ok:
        status.append("B: connected")
    else:
        n_disc = diag['branch_b']['n_disconnected']
        n_comp = diag['branch_b']['n_components']
        status.append(f"B: BROKEN ({n_disc} disconnected px, {n_comp} components)")
    
    color = "green" if (a_ok and b_ok) else "red"
    ax.set_title(f"{title}\n" + " | ".join(status), fontsize=9, color=color)
    ax.set_xticks([])
    ax.set_yticks([])
    return diag


def plot_connectivity_overlay(raster, branch_val, ax=None):
    """Show connected components of a single branch + confluence."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    mask = (raster == branch_val) | (raster == CONFLUENCE_MARKER)
    labeled, n_comp = ndimage.label(mask, structure=STRUCT_8)
    
    # Show labeled components with distinct colors
    display = np.zeros_like(raster, dtype=float)
    display[:] = np.nan
    for c in range(1, n_comp + 1):
        display[labeled == c] = c
    
    ax.imshow(raster, cmap=CLASS_CMAP, vmin=0, vmax=4, interpolation="nearest", alpha=0.3)
    ax.imshow(display, cmap="tab10", interpolation="nearest", alpha=0.8)
    branch_name = "A" if branch_val == BRANCH_A else "B"
    ax.set_title(f"Branch {branch_name}: {n_comp} component(s)", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])


print("Visualization utilities loaded.")

## 3. Load pre-computed rasters and check connectivity

In [ ]:
import pandas as pd
from channel_heads.geometric_analysis import default_stream_loader
from channel_heads.basin_config import get_basin_config

MASTER_CSV = RESULTS_DIR / "master_dataset_v2.csv"
manifest_path = RESULTS_DIR / "raster_manifest.csv"

# Try loading pre-computed rasters; if not available, rasterize a sample on the fly
df_valid = None
stream_cache = {}

if manifest_path.exists():
    print("Loading existing raster manifest...")
    df_manifest = pd.read_csv(manifest_path)
    df_valid = df_manifest[df_manifest["raster_path"].notna()].copy()
    print(f"  Found {len(df_valid)} rasters in manifest")
    print(f"  Basins: {df_valid['basin'].unique().tolist()}")
else:
    print("No raster_manifest.csv found — will rasterize a sample on the fly.")
    if not MASTER_CSV.exists():
        raise FileNotFoundError(
            f"Master dataset not found at {MASTER_CSV}. "
            "Run notebooks/ml/01_prepare_dataset.ipynb first."
        )

In [ ]:
# If no manifest, rasterize a sample on the fly from the master dataset
if df_valid is None:
    df_master = pd.read_csv(MASTER_CSV)
    # Sample up to 50 pairs per basin (or all if fewer)
    MAX_PER_BASIN = 50
    sampled = df_master.groupby("basin").apply(
        lambda g: g.sample(min(len(g), MAX_PER_BASIN), random_state=42),
        include_groups=False,
    ).reset_index(level=0)
    
    print(f"Rasterizing {len(sampled)} sample pairs across {sampled['basin'].nunique()} basins...")
    
    results_rows = []
    for basin, basin_df in sampled.groupby("basin"):
        config = get_basin_config(basin)
        loaded = default_stream_loader(basin, config["lat"], config["z_th"], 300)
        if loaded is None:
            print(f"  Skipping {basin}: DEM not found")
            continue
        s, dem = loaded
        stream_cache[basin] = (s, dem)
        grid_shape = dem.shape if hasattr(dem, "shape") else dem.z.shape
        
        for _, row in basin_df.iterrows():
            try:
                raster = rasterize_outlet_pair(
                    s, int(row["outlet"]), int(row["head_1"]), int(row["head_2"]),
                    int(row["confluence"]), grid_shape, target_size=128,
                )
                results_rows.append({**row.to_dict(), "_raster": raster})
            except Exception as e:
                pass  # skip failures silently
    
    print(f"  Successfully rasterized {len(results_rows)} pairs")
    
    # Build df_valid with in-memory rasters
    df_valid = pd.DataFrame(results_rows)
    _rasters = {i: r["_raster"] for i, r in enumerate(results_rows)}
    USE_MEMORY_RASTERS = True
else:
    USE_MEMORY_RASTERS = False

print(f"\nTotal pairs for diagnostics: {len(df_valid)}")

In [ ]:
# Run connectivity diagnostics on all rasters

def _load_raster(idx, row):
    """Load raster from memory or disk."""
    if USE_MEMORY_RASTERS:
        return _rasters[idx]
    rpath = Path(row["raster_path"])
    if not rpath.is_absolute():
        rpath = RESULTS_DIR / rpath
    if not rpath.exists():
        return None
    return np.load(rpath)

results = []
for idx, row in df_valid.iterrows():
    raster = _load_raster(idx, row)
    if raster is None:
        continue
    diag = diagnose_raster(raster)
    results.append({
        "basin": row.get("basin", "unknown"),
        "outlet": row.get("outlet", -1),
        "head_1": row.get("head_1", -1),
        "head_2": row.get("head_2", -1),
        "confluence": row.get("confluence", -1),
        "raster_idx": idx,
        "a_connected": diag["branch_a"]["connected"],
        "a_n_components": diag["branch_a"]["n_components"],
        "a_n_branch_px": diag["branch_a"]["n_branch_pixels"],
        "a_n_disconnected": diag["branch_a"]["n_disconnected"],
        "b_connected": diag["branch_b"]["connected"],
        "b_n_components": diag["branch_b"]["n_components"],
        "b_n_branch_px": diag["branch_b"]["n_branch_pixels"],
        "b_n_disconnected": diag["branch_b"]["n_disconnected"],
        "has_confluence": diag["has_confluence"],
    })

df_diag = pd.DataFrame(results)
df_diag["both_connected"] = df_diag["a_connected"] & df_diag["b_connected"]
df_diag["any_broken"] = ~df_diag["both_connected"]
print(f"Analyzed {len(df_diag)} rasters")

## 4. Summary statistics

In [ ]:
n_total = len(df_diag)
n_both_ok = df_diag["both_connected"].sum()
n_a_broken = (~df_diag["a_connected"]).sum()
n_b_broken = (~df_diag["b_connected"]).sum()
n_any_broken = df_diag["any_broken"].sum()
n_no_conf = (~df_diag["has_confluence"]).sum()

print("=" * 50)
print("CONNECTIVITY SUMMARY")
print("=" * 50)
print(f"Total rasters:              {n_total:>6}")
print(f"Both branches connected:    {n_both_ok:>6}  ({100*n_both_ok/n_total:.1f}%)")
print(f"Any branch broken:          {n_any_broken:>6}  ({100*n_any_broken/n_total:.1f}%)")
print(f"  Branch A broken:          {n_a_broken:>6}  ({100*n_a_broken/n_total:.1f}%)")
print(f"  Branch B broken:          {n_b_broken:>6}  ({100*n_b_broken/n_total:.1f}%)")
print(f"Missing confluence marker:  {n_no_conf:>6}  ({100*n_no_conf/n_total:.1f}%)")
print()

# Per-basin breakdown
print("Per-basin breakdown (% with any broken branch):")
basin_stats = df_diag.groupby("basin").agg(
    total=("any_broken", "count"),
    n_broken=("any_broken", "sum"),
).assign(pct_broken=lambda x: 100 * x["n_broken"] / x["total"])
basin_stats = basin_stats.sort_values("pct_broken", ascending=False)
for basin, row in basin_stats.iterrows():
    print(f"  {basin:<25s} {row['n_broken']:>4.0f}/{row['total']:>4.0f}  ({row['pct_broken']:.0f}%)")

## 5. Visualize worst cases

In [ ]:
# Show a few broken rasters with connectivity overlay
df_broken = df_diag[df_diag["any_broken"]].copy()
# Sort by most disconnected pixels
df_broken["total_disconnected"] = df_broken["a_n_disconnected"] + df_broken["b_n_disconnected"]
df_broken = df_broken.sort_values("total_disconnected", ascending=False)

n_show = min(8, len(df_broken))
if n_show == 0:
    print("No broken rasters found! The Bresenham fix is working.")
else:
    fig, axes = plt.subplots(2, n_show, figsize=(3.5 * n_show, 7))
    if n_show == 1:
        axes = axes.reshape(2, 1)
    
    for i in range(n_show):
        row = df_broken.iloc[i]
        raster = _load_raster(row["raster_idx"], row)
        title = f"{row['basin']}\no={row['outlet']} h={row['head_1']},{row['head_2']}"
        
        plot_raster_with_diagnostics(raster, title=title, ax=axes[0, i])
        
        # Show whichever branch is more broken
        worse_branch = BRANCH_A if row["a_n_disconnected"] >= row["b_n_disconnected"] else BRANCH_B
        plot_connectivity_overlay(raster, worse_branch, ax=axes[1, i])
    
    fig.suptitle("Worst connectivity failures", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Show a sample of rasters (connected or not) for visual inspection
n_sample = min(8, len(df_diag))
sample_rows = df_diag.sample(n_sample, random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(1, n_sample, figsize=(3.5 * n_sample, 4))
if n_sample == 1:
    axes = [axes]

for i in range(n_sample):
    row = sample_rows.iloc[i]
    raster = _load_raster(row["raster_idx"], row)
    title = f"{row['basin']}\no={int(row['outlet'])} h={int(row['head_1'])},{int(row['head_2'])}"
    plot_raster_with_diagnostics(raster, title=title, ax=axes[i])

fig.suptitle("Sample raster patches (after Bresenham fix)", fontsize=14, y=1.02)
fig.legend(handles=LEGEND_PATCHES, loc="lower center", ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.show()

## 6. Root cause analysis: pre-resize vs post-resize

Is the fragmentation caused by:
- **(a)** Rotation + rounding to integer pixels (gaps in the temp grid before resize), or
- **(b)** Nearest-neighbor resize destroying adjacency?

We test by rasterizing at native resolution (no resize) and checking connectivity.

In [ ]:
# Pick a few broken examples and rasterize at native resolution
n_check = min(6, len(df_broken))
if n_check == 0:
    print("No broken rasters to analyze.")
else:
    fig, axes = plt.subplots(2, n_check, figsize=(4 * n_check, 8))
    if n_check == 1:
        axes = axes.reshape(2, 1)
    
    for i in range(n_check):
        row = df_broken.iloc[i]
        basin = row["basin"]
        
        # Load stream network
        if basin not in stream_cache:
            config = get_basin_config(basin)
            result = default_stream_loader(basin, config["lat"], config["z_th"], 300)
            if result is None:
                print(f"  Skipping {basin}: DEM not found")
                continue
            stream_cache[basin] = result
        
        s, dem = stream_cache[basin]
        grid_shape = dem.shape if hasattr(dem, "shape") else dem.z.shape
        
        outlet = int(row["outlet"])
        head_1 = int(row["head_1"])
        head_2 = int(row["head_2"])
        confluence = int(row["confluence"])
        
        # Rasterize at a very large target size (effectively native resolution)
        native = rasterize_outlet_pair(
            s, outlet, head_1, head_2, confluence, grid_shape,
            target_size=512, padding_frac=0.2,
        )
        resized_128 = rasterize_outlet_pair(
            s, outlet, head_1, head_2, confluence, grid_shape,
            target_size=128, padding_frac=0.2,
        )
        
        title_native = f"{basin} (512x512)\no={outlet} h={head_1},{head_2}"
        title_128 = f"{basin} (128x128)\no={outlet} h={head_1},{head_2}"
        
        plot_raster_with_diagnostics(native, title=title_native, ax=axes[0, i])
        plot_raster_with_diagnostics(resized_128, title=title_128, ax=axes[1, i])
    
    axes[0, 0].set_ylabel("512x512 (near-native)", fontsize=11)
    axes[1, 0].set_ylabel("128x128 (resized)", fontsize=11)
    fig.suptitle("Pre-resize vs post-resize connectivity", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## 7. Gap analysis: distance between consecutive path nodes in raster space

After rotation and rounding, how far apart are consecutive nodes along each path?
If the max Chebyshev distance exceeds 1, there's a gap.

In [ ]:
import math

def measure_path_gaps(s, outlet, head_1, head_2, confluence, grid_shape):
    """Measure pixel gaps between consecutive nodes along each path after rotation."""
    r_nodes, c_nodes = _get_rc(s)
    parents = _build_parents_from_stream(s)
    basin_node_list = _collect_basin_nodes_from_outlet(parents, outlet)
    basin_node_set = set(basin_node_list)
    children = _build_children_from_parents(parents, basin_node_set)
    
    path_a = _trace_full_path(head_1, confluence, children)
    path_b = _trace_full_path(head_2, confluence, children)
    
    conf_r, conf_c = float(r_nodes[confluence]), float(c_nodes[confluence])
    h1_r, h1_c = float(r_nodes[head_1]), float(c_nodes[head_1])
    h2_r, h2_c = float(r_nodes[head_2]), float(c_nodes[head_2])
    
    angle = _compute_rotation_angle(
        head_1_rc=(h1_r, h1_c),
        head_2_rc=(h2_r, h2_c),
        confluence_rc=(conf_r, conf_c),
    )
    
    gap_info = {}
    for label, path in [("A", path_a), ("B", path_b)]:
        if len(path) < 2:
            gap_info[label] = {"max_gap": 0, "mean_gap": 0, "n_gaps": 0, "path_len": len(path)}
            continue
        
        # Get rotated pixel coordinates
        path_r = np.array([float(r_nodes[n]) for n in path])
        path_c = np.array([float(c_nodes[n]) for n in path])
        rot_r, rot_c = _rotate_coordinates(path_r, path_c, conf_r, conf_c, angle)
        
        # Round to integer pixels
        pix_r = np.round(rot_r).astype(int)
        pix_c = np.round(rot_c).astype(int)
        
        # Chebyshev distance between consecutive pixels
        dr = np.abs(np.diff(pix_r))
        dc = np.abs(np.diff(pix_c))
        chebyshev = np.maximum(dr, dc)
        
        gap_info[label] = {
            "max_gap": int(chebyshev.max()),
            "mean_gap": float(chebyshev.mean()),
            "n_gaps": int((chebyshev > 1).sum()),  # gaps = Chebyshev > 1
            "path_len": len(path),
            "gaps_pct": float(100 * (chebyshev > 1).sum() / len(chebyshev)),
        }
    
    return gap_info

print("Gap analysis function loaded.")

In [ ]:
# Run gap analysis on a sample of broken rasters
n_gap_check = min(20, len(df_broken))
gap_results = []

for i in range(n_gap_check):
    row = df_broken.iloc[i]
    basin = row["basin"]
    
    if basin not in stream_cache:
        config = get_basin_config(basin)
        result = default_stream_loader(basin, config["lat"], config["z_th"], 300)
        if result is None:
            continue
        stream_cache[basin] = result
    
    s, dem = stream_cache[basin]
    grid_shape = dem.shape if hasattr(dem, "shape") else dem.z.shape
    
    outlet = int(row["outlet"])
    head_1 = int(row["head_1"])
    head_2 = int(row["head_2"])
    confluence = int(row["confluence"])
    
    try:
        gaps = measure_path_gaps(s, outlet, head_1, head_2, confluence, grid_shape)
        for label in ["A", "B"]:
            gap_results.append({
                "basin": basin, "outlet": outlet, "branch": label,
                **gaps[label]
            })
    except Exception as e:
        print(f"  Error on {basin} o={outlet}: {e}")

df_gaps = pd.DataFrame(gap_results)
if len(df_gaps) > 0:
    print(f"\nGap analysis on {len(df_gaps)//2} broken pairs:")
    print(f"  Max gap (Chebyshev) across all paths: {df_gaps['max_gap'].max()}")
    print(f"  Mean max gap: {df_gaps['max_gap'].mean():.1f}")
    print(f"  Paths with any gap > 1: {(df_gaps['n_gaps'] > 0).sum()}/{len(df_gaps)}")
    print(f"  Mean % of edges with gaps: {df_gaps['gaps_pct'].mean():.1f}%")
    print()
    print("Distribution of max gap per path:")
    print(df_gaps["max_gap"].value_counts().sort_index())
else:
    print("No broken rasters to analyze — all paths are 8-connected!")

In [ ]:
if len(df_gaps) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df_gaps["max_gap"], bins=range(0, df_gaps["max_gap"].max() + 2),
                 edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Max Chebyshev gap between consecutive nodes")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Max gap per path (broken rasters only)")
    axes[0].axvline(1.5, color="red", ls="--", label="8-connected threshold")
    axes[0].legend()

    axes[1].hist(df_gaps["gaps_pct"], bins=20, edgecolor="black", alpha=0.7)
    axes[1].set_xlabel("% of consecutive edges with gap > 1")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Gap prevalence per path")

    plt.tight_layout()
    plt.show()
else:
    print("No gap data to plot — all rasters have valid connectivity.")

## 8. Conclusions

**Root cause:** After rotation, consecutive stream nodes that were 8-connected on the original DEM grid can land more than 1 pixel apart in raster space. `int(round(...))` places each node independently, creating gaps.

**Fix:** Draw Bresenham lines between consecutive nodes along each path to ensure continuous 8-connectivity. This should be applied in `rasterize_outlet_pair()` after rotation, before the resize step.